<a href="https://colab.research.google.com/github/yonialt/Smart-resource-Management-For-Gonder-University-AI-integrated-/blob/main/ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pandas numpy scikit-learn joblib

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import joblib

In [8]:
data = {
    "asset_age_years": [1,2,3,4,5,6,7,8,2,3,6,7,10,12,4,5,3,8,9,10],
    "usage_hours_per_day": [2,3,4,5,6,7,8,9,3,4,6,7,8,9,5,6,4,7,8,9],
    "maintenance_count": [0,1,1,2,2,3,3,4,1,1,2,3,4,5,2,2,1,3,4,5],
    "failure_history": [0,0,1,1,1,1,1,1,0,1,1,1,1,1,0,1,0,1,1,1],
    "condition_score": [90,85,80,70,60,55,50,40,88,82,65,58,45,35,75,72,84,50,48,42],
    "risk_level": [
        "Low","Low","Medium","Medium","High","High","High","High",
        "Low","Medium","High","High","High","High","Medium","Medium",
        "Low","High","High","High"
    ],
    "action": [
        "Keep","Keep","Monitor","Repair","Repair","Repair","Replace","Replace",
        "Keep","Monitor","Repair","Repair","Replace","Replace","Monitor","Repair",
        "Keep","Repair","Replace","Replace"
    ]
}

df = pd.DataFrame(data)
df.head()

,asset_age_years,usage_hours_per_day,maintenance_count,failure_history,condition_score,risk_level,action
0,1,2,0,0,90,Low,Keep
1,2,3,1,0,85,Low,Keep
2,3,4,1,1,80,Medium,Monitor
3,4,5,2,1,70,Medium,Repair
4,5,6,2,1,60,High,Repair


In [10]:
le_risk = LabelEncoder()
le_action = LabelEncoder()

df["risk_level"] = le_risk.fit_transform(df["risk_level"])
df["action"] = le_action.fit_transform(df["action"])

X = df.drop(["risk_level", "action"], axis=1)
y_risk = df["risk_level"]
y_action = df["action"]

In [11]:
X_train, X_test, y_risk_train, y_risk_test = train_test_split(
    X, y_risk, test_size=0.2, random_state=42
)

_, _, y_action_train, y_action_test = train_test_split(
    X, y_action, test_size=0.2, random_state=42
)

In [12]:
risk_model = RandomForestClassifier(n_estimators=100, random_state=42)
action_model = RandomForestClassifier(n_estimators=100, random_state=42)

risk_model.fit(X_train, y_risk_train)
action_model.fit(X_train, y_action_train)

RandomForestClassifier(random_state=42)

In [13]:
risk_pred = risk_model.predict(X_test)
action_pred = action_model.predict(X_test)

print("RISK MODEL ACCURACY:", accuracy_score(y_risk_test, risk_pred))
print("ACTION MODEL ACCURACY:", accuracy_score(y_action_test, action_pred))

print("\n--- RISK REPORT ---")
print(classification_report(y_risk_test, risk_pred))

RISK MODEL ACCURACY: 0.75
ACTION MODEL ACCURACY: 0.75

--- RISK REPORT ---
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       1.00      1.00      1.00         2
           2       0.00      0.00      0.00         1

    accuracy                           0.75         4
   macro avg       0.50      0.67      0.56         4
weighted avg       0.62      0.75      0.67         4



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
def predict_resource(asset_age, usage_hours, maintenance_count, failure_history, condition_score):

    input_data = np.array([[asset_age, usage_hours, maintenance_count, failure_history, condition_score]])

    risk = risk_model.predict(input_data)[0]
    action = action_model.predict(input_data)[0]

    return {
        "risk_level": le_risk.inverse_transform([risk])[0],
        "recommended_action": le_action.inverse_transform([action])[0]
    }

In [19]:
result = predict_resource(
    asset_age=6,
    usage_hours=8,
    maintenance_count=3,
    failure_history=1,
    condition_score=55
)

print(result)

{'risk_level': np.int64(0), 'recommended_action': np.int64(2)}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [18]:
joblib.dump(risk_model, "risk_model.pkl")
joblib.dump(action_model, "action_model.pkl")

joblib.dump(le_risk, "risk_encoder.pkl")
joblib.dump(le_action, "action_encoder.pkl")

['action_encoder.pkl']